In [ ]:
import os
import sys
from dotenv import load_dotenv, find_dotenv
load_dotenv(find_dotenv())
backend_path = os.path.abspath(os.path.join(os.getcwd(), '..'))
sys.path.append(backend_path)
from src.database.database import create_session # noqa: E402
from src.database import models # noqa: E402
import nest_asyncio # noqa: E402
nest_asyncio.apply()

In [ ]:
with create_session() as session:
    project_list = session.query(models.Project)\
        .where(models.Project.is_active == True)\
        .all()
    print(f"Found {len(project_list)} active projects.\n")
    for project in project_list:
        print(f"Processing project: {project.id}-{project.name}")
        project_kanban_columns = session.query(models.ProjectKanbanColumn)\
            .where(models.ProjectKanbanColumn.project_id == project.id)\
            .all()
        
        if not project_kanban_columns:
            print(f"No kanban columns found for project {project.name}. Creating default columns.")
            default_columns = [
                "Backlog",
                "Especificando",
                "Validando",
                "Refinando",
                "Desenvolvendo",
                "Testando",
                "Concluido",
            ]
            backlog_column = models.ProjectKanbanColumn(
                name="Backlog", project_id=project.id, position=0
            )
            for order, column_name in enumerate(default_columns):
                if column_name == "Backlog":
                    continue
                column = models.ProjectKanbanColumn(
                    name=column_name, project_id=project.id, position=order
                )
                session.add(column)
            session.add(backlog_column)
            stories = session.query(models.UserStory).where(models.UserStory.project_id == project.id).all()
            for story in stories:
                story.project_kanban_column_id = backlog_column.id
                session.add(story)
            session.commit()

            
        print(f"Finished processing project: {project.name}\n")
    print(f"Processed {len(project_list)} projects.")
